# Helpstroll — Fine-tune Pixtral 12B via Mistral API

Run in Google Colab (CPU/free tier is fine — training runs on Mistral's servers).

In [ ]:
!pip install mistralai python-dotenv -q

In [ ]:
import os
from google.colab import userdata

MISTRAL_API_KEY = userdata.get('MISTRAL_API_KEY')  # Set in Colab Secrets
os.environ['MISTRAL_API_KEY'] = MISTRAL_API_KEY
print('Key set:', bool(MISTRAL_API_KEY))

In [ ]:
# Upload your helpstroll_dataset.jsonl here
from google.colab import files
uploaded = files.upload()  # select helpstroll_dataset.jsonl

In [ ]:
from mistralai import Mistral
import time

client = Mistral(api_key=MISTRAL_API_KEY)

# Upload dataset
dataset_path = list(uploaded.keys())[0]
with open(dataset_path, 'rb') as f:
    upload = client.files.upload(
        file={'file_name': dataset_path, 'content': f},
        purpose='fine-tune'
    )
file_id = upload.id
print('File ID:', file_id)

In [ ]:
# Create fine-tuning job
job = client.fine_tuning.jobs.create(
    model='open-mistral-nemo',
    training_files=[{'file_id': file_id, 'weight': 1}],
    hyperparameters={'training_steps': 100, 'learning_rate': 1e-4},
    suffix='helpstroll',
)
job_id = job.id
print('Job ID:', job_id)

In [ ]:
# Poll until done
while True:
    status = client.fine_tuning.jobs.get(job_id=job_id)
    print('Status:', status.status)
    if status.status in ('succeeded', 'failed', 'cancelled'):
        break
    time.sleep(30)

model_id = status.fine_tuned_model
print('Fine-tuned model ID:', model_id)

In [ ]:
# Test inference
resp = client.chat.complete(
    model=model_id,
    messages=[{'role': 'user', 'content': [
        {'type': 'text', 'text': 'Is this person in danger? Respond DISTRESS or SAFE.'},
        {'type': 'image_url', 'image_url': {'url': 'https://images.unsplash.com/photo-1507003211169-0a1dd7228f2d?w=320'}},
    ]}],
    max_tokens=5,
)
print('Model says:', resp.choices[0].message.content)